# Extended Data Figure 1 and related information

This notebook computes the golden sets used in Proof-of-Concept study (6-nt) and the experiments (7-nt).

Following the hybrid initial point selection paradigm [41], we construct a representative training set from the vast space of potential PAM candidates by enumerating every k-mer of the specified PAM length and encoding each as a node in a phylogenetic tree that captures sequence similarity.

In [1]:
import os
import itertools
import numpy as np
import pandas as pd
import os.path as op
from tqdm import tqdm
from Bio import motifs
from Bio.Seq import Seq
import Levenshtein as lev
from Bio.Data import IUPACData
import matplotlib.pyplot as plt
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

In [2]:
def generate_pams(n=6):
    """
    Generates all possible DNA PAM sequences of length `n`.
    Constructs a DataFrame containing all combinations of the DNA bases A, T, G, C.
    Returns a Pandas DataFrame with the generated sequences in the 'PAM' column.
    """
    base = ["ATGC"]*n
    df = pd.DataFrame({"PAM": np.array(["".join(a) for a in itertools.product(*base)])})
    return(df)

In [3]:
def golden_set(df, threshold=4, max_threshold=6):
    """
    Computes the Golden Set for PAM candidates.
    
    Performs hierarchical clustering on PAM sequences based on Levenshtein distance.
    Calculates the distance matrix and linkage, then determines cluster counts at various thresholds.
    Identifies clusters, computes consensus sequences, and returns the clusters sorted by distance to consensus.
    """
    levn = np.zeros((df.shape[0], df.shape[0]))
    #The tree is built using a standard distance metric (Levenshtein distance) 
    #and a hierarchical clustering algorithm.
    print("Computing distance matrix")
    for i,a in tqdm(enumerate(df["PAM"])):
        for j,b in enumerate(df["PAM"]):
            levn[i,j] = lev.distance(a,b)
    print("Computing average linkage")
    lnk = linkage(squareform(levn), method="average")
    n_clusts = []
    print("Calculating number of clusters")
    for t in tqdm(np.arange(0,max_threshold)):
        n_clusts.append(
            np.unique(fcluster(lnk, t, criterion="distance")).shape[0]
        )
    #Once the tree is assembled, we prune it according to a user-defined criterion that reflects
    #the desired size of the initial training set.
    print("Cutting the three at threshold="+str(threshold))
    eight = fcluster(lnk, threshold, criterion="distance")
    clusters = {}
    print("Clustering PAMs")
    cluster_numbers = np.unique(eight)
    for a in tqdm(np.arange(1,np.max(cluster_numbers)+1)):
        PAM = str(motifs.create([Seq(a) for a in df["PAM"][eight == a].values]).consensus)
        clusters[PAM] = df["PAM"][eight == a].values
    sorted_clusters = {}
    print("Sorting clusters")
    for a in tqdm(clusters):
        sorted_clusters[a] = sorted(clusters[a], key=lambda x: lev.distance(x, a))
    return(sorted_clusters, levn, lnk)

# Generate the Golden Set for 6-nt PAM candidates

In [4]:
sorted_clusters_F2, levn_F2, lnk_F2 = golden_set(generate_pams(6), 4, 6)

Computing distance matrix


4096it [00:09, 435.28it/s]


Computing average linkage
Calculating number of clusters


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:00<00:00, 167.67it/s]


Cutting the three at threshold=4
Clustering PAMs


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:00<00:00, 136.74it/s]


Sorting clusters


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:00<00:00, 3638.91it/s]


After pruning, we identify the centroids of each flat cluster, which serve as representative “center” points, and also select “border” points that lie at the extremities of the clusters.

In [5]:
golden_set_F2 = []
for a in sorted_clusters_F2:
    golden_set_F2.extend(sorted_clusters_F2[a][0:10]) #The center points
    golden_set_F2.extend(sorted_clusters_F2[a][-15:]) #The boundary points

In our Proof-of-Concept experiments, we used 180 PAM candidates (ten experimental rounds; see Supplementary Data SD2) total as the Golden Set.

In [6]:
len(golden_set_F2)

200

In [7]:
with open("../data/processed/GS_6nt.fasta", "w") as oh:
    for i,a in enumerate(golden_set_F2[0:181]):
        oh.write(">"+str(i)+"\n")
        oh.write(a+"\n")

# Generate the Golden Set for 7-nt PAM candidates

In [ ]:
sorted_clusters_F3, levn_F3, lnk_F3 = golden_set(generate_pams(7), 4, 6)

Computing distance matrix


11412it [01:45, 98.74it/s]

In [ ]:
golden_set_F3 = []
for a in sorted_clusters_F3:#Center points
    golden_set_F3.extend([sorted_clusters_F3[a][0]])
for a in sorted_clusters_F3:#Boundary points
    golden_set_F3.extend([sorted_clusters_F3[a][-1]])

In [ ]:
len(golden_set_F3)

To get rid of experimental problems caused by long T strings, we filter the golden set to exclude every string that has 4 and more consecutive Ts.

In [ ]:
golden_set_F3 = list(filter(lambda x: "TTTT" not in x, golden_set_F3))

In [ ]:
len(golden_set_F3)

For experiments with SpCas9, PpCas9 and AlkaCas9, we use 72 PAM candidates (four experimental rounds; see Supplementary Data SD3) as switching round protocol with switching at fifth experimental round requires.

In [ ]:
with open("../data/processed/GS_7nt.fasta", "w") as oh:
    for i,a in enumerate(golden_set_F3[0:72]):
        oh.write(">"+str(i)+"\n")
        oh.write(a+"\n")

# Plot the supplementary figure SF1

In [ ]:
plt.rcParams["figure.figsize"] = 13,15
plt.rcParams["figure.dpi"] = 300


#fig, ax = plt.subplots()
fig = plt.figure()
fig.tight_layout()
axd = fig.subplot_mosaic(
    """
    ABB
    CCC
    DEE
    FFF
    """
)
axd["A"].imshow(levn_F2)
axd["A"].set_title("a", loc="left", weight="bold")
n_clusts_F2 = [np.unique(fcluster(lnk_F2, t, criterion="distance")).shape[0] for t in np.arange(1,7)]
axd["B"].plot(np.arange(1,7), n_clusts_F2)
axd["B"].scatter(np.arange(1,7), n_clusts_F2)
for a,b in zip(np.arange(1,7), n_clusts_F2):
    axd["B"].annotate(xy=(a,b), text=str(b), weight="bold" if b == 8 else "normal")
axd["B"].set_xlabel("Levenshtein distance threshold")
axd["B"].set_ylabel("Number of unique clusters")
axd["B"].set_title("b", loc="left", weight="bold")
u_F2 = dendrogram(lnk_F2, truncate_mode="level", p=5, ax=axd["C"])
axd["C"].set_title("c", loc="left", weight="bold")
axd["C"].axhline(y=4, color="red", linestyle="dashed")
axd["C"].set_ylabel("Levenshtein distance threshold")
axd["D"].imshow(levn_F3)
axd["D"].set_title("d", loc="left", weight="bold")
n_clusts_F3 = [np.unique(fcluster(lnk_F3, t, criterion="distance")).shape[0] for t in np.arange(1,8)]
axd["E"].plot(np.arange(1,8), n_clusts_F3)
axd["E"].scatter(np.arange(1,8), n_clusts_F3)
for a,b in zip(np.arange(1,8), n_clusts_F3):
    axd["E"].annotate(xy=(a,b), text=str(b), weight="bold" if b == 63 else "normal")
axd["E"].set_title("e", loc="left", weight="bold")
axd["E"].set_xlabel("Levenshtein distance threshold")
axd["E"].set_ylabel("Number of unique clusters")
u_F3 = dendrogram(lnk_F3, truncate_mode="level", p=5, ax=axd["F"])
axd["F"].axhline(y=4, color="red", linestyle="dashed")
axd["F"].set_title("f", loc="left", weight="bold")
axd["F"].set_ylabel("Levenshtein distance threshold")
plt.savefig("EDF1.png")
plt.show()

# Test for whether the golden set already contains PAM

In [ ]:
def matches_motif(sequence, motif="NNNNRTN"):
    """
    Check if a DNA sequence matches the motif, supporting ambiguous IUPAC codes.

    The motif is defined as:
        - positions 1–4: any nucleotide (N) – compatible with ALL ambiguous bases
        - position 5:     purine (R) – must be compatible with A or G
        - position 6:     thymine (T) – must be compatible with T

    Parameters
    ----------
    sequence : str or Bio.Seq.Seq
        Input DNA sequence (may contain ambiguous IUPAC letters).
    motif : str
        Motif to match with

    Returns
    -------
    bool
        True if the sequence exactly matches the motif (length 6 and each position
        is compatible with the motif's base), False otherwise.
    """
    # Convert to string and uppercase
    seq_str = str(sequence).upper()
    motif = [a for a in motif]
    # Get the IUPAC ambiguity mapping: each letter -> set of possible unambiguous bases
    ambig_map = IUPACData.ambiguous_dna_values
    for seq_base, motif_base in zip(seq_str, motif):
        # If the sequence character is not a valid IUPAC DNA letter, fail
        if seq_base not in ambig_map:
            return False
        # Get the set of unambiguous bases represented by the sequence character
        seq_set = set(ambig_map[seq_base])
        # Get the set for the motif character
        motif_set = set(ambig_map[motif_base])
        # Compatible if the two sets intersect (they share at least one unambiguous base)
        if not (seq_set & motif_set):
            return False
    # All positions passed compatibility
    return True

In [ ]:
gs_to_check = []
with open("../data/processed/GS_7nt.fasta", "r") as ih:
    for a in ih.readlines():
        if a[0] != ">":
            gs_to_check.append(a.replace("\n", ""))

In [ ]:
ppPAMS = []
for a in gs_to_check:
    if matches_motif(a, "NNNNRTN"):
        ppPAMS.append(a)

In [ ]:
ppPAMS

In [ ]:
spPAMS = []
for a in gs_to_check:
    if matches_motif(a, "NGGNNNN"):
        spPAMS.append(a)

In [ ]:
spPAMS

In [ ]:
alkaPAMS = []
for a in gs_to_check:
    if matches_motif(a, "NNGAVNN"):
        alkaPAMS.append(a)

In [ ]:
alkaPAMS

Yes, for all proteins, there were matches in the PAM golden set, however, we could not have known it for AlkaCas9 beforehand.

# References for this part:

41. Yuan, W., Han, Y., Guan, D., Lee, S. & Lee, Y.-K. for Computing Machinery,
A. (ed.) Initial training data selection for active learning. (ed.for Comput-
ing Machinery, A.) Proceedings of the 5th International Conference on Ubiquitous
Information Management and Communication, ICUIMC ’11 (New York, NY,
USA, 2011). URL https://doi.org/10.1145/1968613.1968619.